# BlobNet Atom Finder

Acquire a HAADF image, run BlobNet to find and save atom positions.

### Run the servers

Make sure you are on the VPN and the AutoScript server is running. Then start the asyncroscopy Tango servers from the repository root:

```bash
uv run startup_scripts/run_servers.py
```


### Imports


In [ ]:
import os
import json
import tango
import numpy as np
from pprint import pprint
import matplotlib.pyplot as plt
from tiled.client import from_uri

%matplotlib ipympl

### Ping servers


In [ ]:
DB_HOST = "10.46.217.241"
# DB_HOST = "127.0.0.1"
DB_PORT = 9094

os.environ["TANGO_HOST"] = f"{DB_HOST}:{DB_PORT}"

scan = tango.DeviceProxy("asyncroscopy/scan/default")
camera = tango.DeviceProxy("asyncroscopy/camera/default")
microscope = tango.DeviceProxy("asyncroscopy/instrument/default")
data = tango.DeviceProxy("asyncroscopy/data/default")

for proxy in [scan, microscope, data]:
    proxy.set_timeout_millis(120_000)
    proxy.ping()
    print(proxy.name(), proxy.state())

In [ ]:
microscope.unblank_beam()

### Set Tiled Client


In [ ]:
config = json.loads(data.get_config())
client = from_uri(config.get("uri"))
print("Tiled keys:", list(client))

### Configure scan


In [ ]:
scan.dwell_time = 1e-6
scan.imsize = 512
scan.scan_region = [0, 0, 1, 1]
scan.output_format = ".h5"

print("dwell_time :", scan.dwell_time)
print("image size :", scan.imsize)
print("scan region:", list(scan.scan_region))


In [ ]:
microscope.set_column_valves('open')

In [ ]:
json.loads(microscope.get_parameters())

### Acquire a HAADF image


In [ ]:
microscope.set_fov(10*1e-9)
microscope.set_image_shift([0e-9, 0])
data_key = microscope.acquire_scanned_image(["haadf"])

image = client[data_key]["image"]["HAADF"].read()
metadata = dict(client[data_key]["image"]["HAADF"].metadata)

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image, cmap="gray", interpolation="none")
ax.set_title(f"HAADF - dwell {scan.dwell_time * 1e6:.1f} us")
ax.axis("off")
plt.tight_layout()

In [ ]:
print("Metadata:")
pprint(metadata)

print("Image shape:", image.shape)
print("Image dtype:", image.dtype)


---
# BlobNet: find the atoms in the HAADF image

Everything below operates on `image` from the cell above.

### Where to run BlobNet

`DEVICE` selects the compute device: `"auto"`, `"cuda"` or `"cpu"`.

In [ ]:
import sys
from pathlib import Path
import torch

BLOBNET_ROOT = Path(r"C:/path/to/BlobNet")                                     # BlobNet repository root
CHECKPOINT   = BLOBNET_ROOT / "outputs/manuscript_models/random/unet_best.pth"  # trained weights
DEVICE       = "auto"                                                          # "auto", "cuda" or "cpu"

NUM_FILTERS = [32, 64, 128, 256]   # must match the checkpoint
DROPOUT = 0.2

if str(BLOBNET_ROOT) not in sys.path:
    sys.path.insert(0, str(BLOBNET_ROOT))

from blobnet import build_unet, extract_subpixel_peak_positions, plot_localization_result

### BlobNet functions

Copied from `BlobNet/scripts/make_manuscript_figures.py` so behaviour matches the
reference implementation.

In [ ]:
def _device_from_name(name: str) -> torch.device:
    if name != 'auto':
        return torch.device(name)
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    if torch.cuda.is_available():
        return torch.device('cuda')
    return torch.device('cpu')


def _normalize_image(image: np.ndarray, low: float = 1.0, high: float = 99.8) -> np.ndarray:
    image = np.asarray(image, dtype=np.float32)
    lo, hi = np.percentile(image, [low, high])
    image = np.clip((image - lo) / max(float(hi - lo), 1e-8), 0.0, 1.0)
    return image.astype(np.float32)


def _load_blobnet_model(
    checkpoint: Path,
    device: torch.device,
    num_filters: list[int],
    dropout: float,
) -> torch.nn.Module:
    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint}')
    model = build_unet(input_channels=1, num_classes=1, num_filters=num_filters, dropout=dropout)
    try:
        payload = torch.load(checkpoint, map_location='cpu', weights_only=False)
    except TypeError:
        payload = torch.load(checkpoint, map_location='cpu')
    state_dict = payload['model_state_dict'] if isinstance(payload, dict) and 'model_state_dict' in payload else payload
    model.load_state_dict(state_dict)
    model.to(device)
    model.eval()
    return model


def _predict_array(model: torch.nn.Module, image: np.ndarray, device: torch.device) -> np.ndarray:
    tensor = torch.from_numpy(np.asarray(image, dtype=np.float32)).unsqueeze(0).unsqueeze(0).to(device)
    with torch.inference_mode():
        output = torch.sigmoid(model(tensor))[0, 0].detach().cpu().numpy()
    return output.astype(np.float32)

### Load the trained network

In [ ]:
device = _device_from_name(DEVICE)
model = _load_blobnet_model(CHECKPOINT, device, num_filters=NUM_FILTERS, dropout=DROPOUT)

print("running on:", device)
if device.type == "cuda":
    print("gpu       :", torch.cuda.get_device_name(0))

### Find the atoms

`find_atoms` normalises the image, predicts a heat map, and returns peak positions.

The network halves the image three times and rebuilds it, so both dimensions must be
divisible by 8.

In [ ]:
THRESHOLD_REL = 0.35   # peak cut-off, relative to the heat map maximum


def find_atoms(
    image: np.ndarray,
    model: torch.nn.Module,
    device: torch.device,
    threshold_rel: float = 0.35,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (normalised image, heat map, atom positions).

    Positions are (row, column) in pixels, sub-pixel refined.
    """
    norm = _normalize_image(np.squeeze(image))
    if norm.ndim != 2:
        raise ValueError(f"Expected a 2-D image, got shape {norm.shape}")
    if norm.shape[0] % 8 or norm.shape[1] % 8:
        raise ValueError(f"Image dimensions must be divisible by 8, got {norm.shape}")

    heat = _predict_array(model, norm, device)
    yx = extract_subpixel_peak_positions(
        heat, threshold_rel=threshold_rel, min_distance=3, window_size=5
    )
    return norm, heat, yx


norm, heat, yx = find_atoms(image, model, device, threshold_rel=THRESHOLD_REL)
print(f"{len(yx)} atoms found in a {norm.shape} image")
print(f"heat map range: {heat.min():.3f} to {heat.max():.3f}")

### See what BlobNet did

Visualize image, the heat map, and the atoms it picked out. 

In [ ]:
fig, axes = plot_localization_result(norm, heat, np.zeros((0, 2)), yx, figsize=(15, 5))
fig.suptitle(f"BlobNet  |  {len(yx)} atoms  |  {device}", y=1.02)

### How sure was it?

Brighter heat-map peaks mean the network was more confident. A long tail of faint
peaks near the cut-off usually means `THRESHOLD_REL` is letting in noise.

In [ ]:
rows = np.clip(np.rint(yx[:, 0]).astype(int), 0, heat.shape[0] - 1)
cols = np.clip(np.rint(yx[:, 1]).astype(int), 0, heat.shape[1] - 1)
peak_height = heat[rows, cols]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(peak_height, bins=40, color="tab:orange")
ax[0].axvline(THRESHOLD_REL * heat.max(), color="k", ls="--", label="cut-off")
ax[0].set_xlabel("peak height (confidence)")
ax[0].set_ylabel("atoms")
ax[0].legend()

ax[1].imshow(norm, cmap="gray")
sc = ax[1].scatter(yx[:, 1], yx[:, 0], c=peak_height, s=10, cmap="viridis")
fig.colorbar(sc, ax=ax[1], label="confidence")
ax[1].set_title("atoms coloured by confidence")
ax[1].axis("off")
plt.tight_layout()

### Save the atom positions

In [ ]:
out_path = Path(f"{data_key}_atoms.csv")
np.savetxt(out_path, yx, delimiter=",", header="row_px,col_px", comments="")
print("saved", out_path.resolve())